In [1]:
import torch

print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built())

True
True


In [3]:
import sys
sys.path.insert(0, '../..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import time
import json
import joblib
import mlflow
from pathlib import Path
from tqdm import tqdm

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    SparseVectorParams, SparseIndexParams,
    SparseVector, NamedVector, NamedSparseVector,
    SearchRequest, Filter, FieldCondition,
    MatchValue, OptimizersConfigDiff,
    HnswConfigDiff
)
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp

from src.utils.config import settings

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


PROC = '../../data/processed/'
FEAT = '../../data/features/'

print(f"✅ Imports ready")
print(f"   Device : {device}")

✅ Imports ready
   Device : mps


In [16]:
import numpy as np

embeddings = np.load(FEAT + "e5_large_embeddings.npy")
EMBED_DIM = embeddings.shape[1]

print("Embeddings:", embeddings.shape)
print("EMBED_DIM:", EMBED_DIM)

Embeddings: (45454, 1024)
EMBED_DIM: 1024


In [17]:
client = QdrantClient(
    host = settings.QDRANT_HOST,
    port = settings.QDRANT_PORT,
)

# Test connection
collections = client.get_collections()
print(f"✅ Connected to Qdrant")
print(f"   Host        : {settings.QDRANT_HOST}:"
      f"{settings.QDRANT_PORT}")
print(f"   Collections : "
      f"{len(collections.collections)}")

# Qdrant version info
print(f"\n   Dashboard   : "
      f"http://localhost:6333/dashboard")

✅ Connected to Qdrant
   Host        : localhost:6333
   Collections : 0

   Dashboard   : http://localhost:6333/dashboard


In [18]:
# Load Data
movies  = pd.read_csv(PROC + 'movies_master.csv',
                      low_memory=False)
ratings = pd.read_csv(PROC + 'ratings_cleaned.csv')

# Parse list columns
import ast
def safe_parse(val):
    try:
        r = ast.literal_eval(str(val))
        return r if isinstance(r, list) else []
    except:
        return []

movies['genres_list']  = movies['genres_list']\
    .apply(safe_parse)
movies['cast_names']   = movies['cast_names']\
    .apply(safe_parse)
movies['keyword_list'] = movies['keyword_list']\
    .apply(safe_parse)

# Fill text fields
for col in ['title', 'overview', 'tagline',
            'director']:
    movies[col] = movies[col].fillna('')

# Only keep movies with movieId
movies = movies[movies['movieId'].notna()].copy()
movies['movieId'] = movies['movieId'].astype(int)

# Load movie interaction stats
mov_stats = pd.read_csv(
    FEAT + 'movie_interaction_features.csv')
movies = movies.merge(
    mov_stats[['movieId', 'rating_mean',
               'rating_count', 'popularity_tier']],
    on='movieId', how='left'
)

print(f"Movies to index : {len(movies):,}")
print(f"Columns         : {list(movies.columns)}")

Movies to index : 45,454
Columns         : ['id', 'title', 'original_title', 'overview', 'tagline', 'genres', 'release_date', 'year', 'original_language', 'budget', 'revenue', 'runtime', 'vote_average', 'vote_count', 'popularity', 'production_companies', 'poster_path', 'imdb_id', 'cast_names', 'director', 'keyword_list', 'movieId', 'tmdbId', 'genres_list', 'rating_mean', 'rating_count', 'popularity_tier']


In [19]:
# Build Text Soup For Embeddings

def build_text_soup(row) -> str:
    """
    Rich text representation per movie.
    Same as Day 4 but optimised for e5-large.
    e5-large works best with natural sentences
    not just keyword concatenation.
    """
    parts = []

    # Title — most important
    if row['title']:
        parts.append(f"Movie: {row['title']}.")

    # Overview — semantic content
    if row['overview']:
        parts.append(row['overview'])

    # Genres as sentence
    if row['genres_list']:
        genres = ', '.join(row['genres_list'])
        parts.append(f"Genre: {genres}.")

    # Director
    if row['director']:
        parts.append(
            f"Directed by {row['director']}.")

    # Cast
    if row['cast_names']:
        cast = ', '.join(row['cast_names'][:3])
        parts.append(f"Starring {cast}.")

    # Keywords
    if row['keyword_list']:
        kw = ', '.join(row['keyword_list'][:8])
        parts.append(f"Keywords: {kw}.")

    # Tagline
    if row['tagline']:
        parts.append(row['tagline'])

    return ' '.join(parts).strip()


movies['text_soup'] = movies.apply(
    build_text_soup, axis=1)

print(f"✅ Text soups built")
print(f"\nExample for '{movies['title'].iloc[0]}':")
print(movies['text_soup'].iloc[0][:400])
print(f"\nAvg length: "
      f"{movies['text_soup'].str.len().mean():.0f} "
      f"chars")

✅ Text soups built

Example for 'Toy Story':
Movie: Toy Story. Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. Genre: Animation, Comedy, Family. Directed by John Lasseter. Starring Tom Hank

Avg length: 505 chars


In [20]:
# Load sentence-transformers e5-large
print("Loading sentence-transformers e5-large...")
print("First run downloads ~1.2GB model")
print("Subsequent runs load from cache\n")

start = time.time()

# e5-large: 2025-26 SOTA for text retrieval
# Better than all-MiniLM on MTEB benchmark
# Use 'query:' prefix for queries
# Use 'passage:' prefix for documents
model_name = 'intfloat/e5-large-v2'

embedder = SentenceTransformer(
    model_name, device=device)

elapsed = time.time() - start
print(f"✅ Model loaded in {elapsed:.1f}s")
print(f"   Model      : {model_name}")
print(f"   Embed dim  : "
      f"{embedder.get_sentence_embedding_dimension()}")
print(f"   Device     : {device}")

# Test embedding
test_emb = embedder.encode(
    ["passage: Test movie about space exploration"],
    normalize_embeddings=True
)
print(f"   Test embed : {test_emb.shape}")

Loading sentence-transformers e5-large...
First run downloads ~1.2GB model
Subsequent runs load from cache

✅ Model loaded in 5.7s
   Model      : intfloat/e5-large-v2
   Embed dim  : 1024
   Device     : mps
   Test embed : (1, 1024)


In [ ]:
movies[["text_soup"]].to_csv(
    "movies_text_soup.csv",
    index=False
)

print("Saved!")

In [8]:
import os
print(os.getcwd())

/Users/lucifer/Desktop/Wilfrid /Courses/Spring 2026/CP612-va2/Group Project/Production-recsys/notebooks/week2_retrieval


In [ ]:
# Generate Dense Embeddings
EMBED_DIM  = embedder\
    .get_embedding_dimension()
BATCH_SIZE = 64

print(f"Generating e5-large embeddings...")
print(f"  Movies     : {len(movies):,}")
print(f"  Embed dim  : {EMBED_DIM}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Device     : {device}\n")

# e5-large uses 'passage:' prefix for documents
texts = [
    f"passage: {soup}"
    for soup in movies['text_soup'].values
]

start      = time.time()
embeddings = embedder.encode(
    texts,
    batch_size        = BATCH_SIZE,
    show_progress_bar = True,
    normalize_embeddings = True,   # L2 normalise
    convert_to_numpy  = True,
)
elapsed = time.time() - start

print(f"\n✅ Embeddings generated")
print(f"   Shape   : {embeddings.shape}")
print(f"   Time    : {elapsed:.1f}s")
print(f"   Dtype   : {embeddings.dtype}")
print(f"   Norm    : "
      f"{np.linalg.norm(embeddings[0]):.4f} "
      f"(should be ~1.0)")

# Save embeddings
np.save(FEAT + 'e5_large_embeddings.npy',
        embeddings)
print(f"\n✅ Embeddings saved to "
      f"data/features/e5_large_embeddings.npy")

In [21]:
# Build BM25 Sparse Vectors
print("Building BM25 sparse vectors...")
print("Used for hybrid search — exact keyword "
      "matching alongside dense vectors\n")

# TF-IDF as BM25 approximation
# Qdrant accepts sparse vectors in BM25 format
tfidf = TfidfVectorizer(
    max_features = 30_000,
    ngram_range  = (1, 2),
    min_df       = 2,
    max_df       = 0.95,
    sublinear_tf = True,
    strip_accents= 'unicode',
)

tfidf_matrix = tfidf.fit_transform(
    movies['text_soup'].values)

print(f"✅ Sparse vectors built")
print(f"   Shape       : {tfidf_matrix.shape}")
print(f"   Vocabulary  : "
      f"{len(tfidf.vocabulary_):,} terms")
print(
    f"   Density     : "
    f"{(tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]) * 100):.3f}%"
)
# Save TF-IDF
joblib.dump(tfidf,
    FEAT + 'tfidf_vectorizer_qdrant.joblib')
sp.save_npz(FEAT + 'tfidf_sparse.npz',
            tfidf_matrix)

print(f"✅ Sparse vectors saved")


Building BM25 sparse vectors...
Used for hybrid search — exact keyword matching alongside dense vectors

✅ Sparse vectors built
   Shape       : (45454, 30000)
   Vocabulary  : 30,000 terms
   Density     : 0.275%
✅ Sparse vectors saved


In [22]:
#Create Qdrant Collection

COLLECTION_NAME = "movies_hybrid"

# Delete if exists — fresh start
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing collection: "
          f"{COLLECTION_NAME}")

# Create collection with:
# 1. Dense vectors (e5-large)
# 2. Sparse vectors (BM25/TF-IDF)
client.create_collection(
    collection_name = COLLECTION_NAME,
    vectors_config  = {
        "dense": VectorParams(
            size     = EMBED_DIM,
            distance = Distance.COSINE,
        )
    },
    sparse_vectors_config = {
        "sparse": SparseVectorParams(
            index = SparseIndexParams(
                on_disk = False
            )
        )
    },
    # HNSW config — controls index quality
    hnsw_config = HnswConfigDiff(
        m              = 16,   # connections per node
        ef_construct   = 100,  # build quality
        full_scan_threshold = 10_000,
    ),
    # Optimiser config
    optimizers_config = OptimizersConfigDiff(
        indexing_threshold = 0,  # index immediately
    ),
)

print(f"✅ Collection created: {COLLECTION_NAME}")
print(f"""
Collection config:
  Dense vectors  : e5-large ({EMBED_DIM} dims)
  Sparse vectors : BM25/TF-IDF (30K vocab)
  Index          : HNSW (m=16, ef=100)
  Distance       : Cosine
  Hybrid search  : dense + sparse simultaneously
""")

✅ Collection created: movies_hybrid

Collection config:
  Dense vectors  : e5-large (1024 dims)
  Sparse vectors : BM25/TF-IDF (30K vocab)
  Index          : HNSW (m=16, ef=100)
  Distance       : Cosine
  Hybrid search  : dense + sparse simultaneously



In [23]:
#Upload Vectors To Qdrant

print(f"Uploading {len(movies):,} movies "
      f"to Qdrant...")
print(f"Each movie gets:")
print(f"  Dense vector  : {EMBED_DIM} floats")
print(f"  Sparse vector : BM25 indices + weights")
print(f"  Payload       : metadata for filtering\n")

UPLOAD_BATCH = 100
n_uploaded   = 0
n_failed     = 0

for batch_start in tqdm(
        range(0, len(movies), UPLOAD_BATCH),
        desc="Uploading"):

    batch_end = min(
        batch_start + UPLOAD_BATCH, len(movies))
    batch_movies = movies.iloc[
        batch_start:batch_end]

    points = []

    for local_idx, (df_idx, row) in enumerate(
            batch_movies.iterrows()):

        global_idx  = batch_start + local_idx
        dense_vec   = embeddings[global_idx].tolist()

        # Sparse vector — BM25
        sparse_row  = tfidf_matrix[global_idx]
        sparse_indices = sparse_row.indices.tolist()
        sparse_values  = sparse_row.data.tolist()

        # Payload — metadata for filtering
        payload = {
            "movie_id":   int(row['movieId']),
            "tmdb_id":    int(row['id'])
                          if pd.notna(row['id'])
                          else 0,
            "title":      str(row['title']),
            "genres":     row['genres_list']
                          if isinstance(
                              row['genres_list'],
                              list) else [],
            "year":       int(row['year'])
                          if pd.notna(
                              row.get('year', None))
                          else 0,
            "director":   str(row['director']),
            "rating_mean": float(
                           row['rating_mean'])
                           if pd.notna(
                               row.get(
                                   'rating_mean'))
                           else 0.0,
            "rating_count": int(
                            row['rating_count'])
                            if pd.notna(
                                row.get(
                                    'rating_count'))
                            else 0,
            "popularity_tier": str(
                               row.get(
                                   'popularity_tier',
                                   'cold')),
            "text_soup":  str(
                          row['text_soup'])[:500],
        }

        point = PointStruct(
            id      = global_idx,
            payload = payload,
            vector  = {
                "dense": dense_vec,
                "sparse": SparseVector(
                    indices = sparse_indices,
                    values  = sparse_values,
                )
            }
        )
        points.append(point)

    try:
        client.upsert(
            collection_name = COLLECTION_NAME,
            points          = points,
            wait            = True,
        )
        n_uploaded += len(points)
    except Exception as e:
        print(f"❌ Batch {batch_start} failed: {e}")
        n_failed += len(points)

print(f"\n✅ Upload complete")
print(f"   Uploaded : {n_uploaded:,} movies")
print(f"   Failed   : {n_failed:,}")

# Verify collection
info = client.get_collection(COLLECTION_NAME)
print(f"   Indexed  : "
      f"{info.points_count:,} points in Qdrant")

Uploading 45,454 movies to Qdrant...
Each movie gets:
  Dense vector  : 1024 floats
  Sparse vector : BM25 indices + weights
  Payload       : metadata for filtering



Uploading: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 455/455 [00:30<00:00, 15.04it/s]


✅ Upload complete
   Uploaded : 45,454 movies
   Failed   : 0
   Indexed  : 45,454 points in Qdrant


In [24]:
movies[movies['title'].str.contains(
    'Interstellar',
    case=False,
    na=False
)][['title', 'year']]

,title,year
22891,Interstellar,2014.0


In [25]:
movies[movies["title"].str.contains("Iron Man", case=False, na=False)][
    ["title", "year", "text_soup"]
]

,title,year,text_soup
4429,Tetsuo: The Iron Man,1989.0,Movie: Tetsuo: The Iron Man. Tetsuo: The Iron ...
12600,Iron Man,2008.0,Movie: Iron Man. After being held captive in a...
15170,Iron Man 2,2010.0,Movie: Iron Man 2. With the world now aware of...
20812,The Invincible Iron Man,2007.0,Movie: The Invincible Iron Man. When a cocky i...
20823,Iron Man: Rise of Technovore,2013.0,Movie: Iron Man: Rise of Technovore. Iron Man ...
20848,Iron Man 3,2013.0,Movie: Iron Man 3. When Tony Stark's world is ...
30893,Iron Man & Captain America: Heroes United,2014.0,Movie: Iron Man & Captain America: Heroes Unit...
32884,Iron Man & Hulk: Heroes United,2013.0,Movie: Iron Man & Hulk: Heroes United. The Inv...
34648,Iron Man,1951.0,"Movie: Iron Man. In Coaltown, Pennsylvania, mi..."
41779,Iron Man,1931.0,Movie: Iron Man. Prizefighter Mason loses his ...


In [26]:
# Dense Search
def dense_search(query: str,
                 top_k: int = 10,
                 genre_filter: str = None
                 ) -> pd.DataFrame:
    """
    Semantic search using e5-large embeddings.
    Understands meaning — not just keywords.
    """
    # e5-large uses 'query:' prefix for queries
    query_emb = embedder.encode(
        [f"query: {query}"],
        normalize_embeddings = True,
        convert_to_numpy     = True,
    )[0].tolist()

    # Optional genre filter
    search_filter = None
    if genre_filter:
        search_filter = Filter(
            must=[FieldCondition(
                key   = "genres",
                match = MatchValue(
                    value=genre_filter)
            )]
        )

    results = client.search(
        collection_name = COLLECTION_NAME,
        query_vector    = NamedVector(
            name   = "dense",
            vector = query_emb,
        ),
        limit           = top_k,
        query_filter    = search_filter,
        with_payload    = True,
    )

    return pd.DataFrame([{
        'title':    r.payload['title'],
        'score':    round(r.score, 4),
        'genres':   r.payload['genres'],
        'year':     r.payload['year'],
        'director': r.payload['director'],
    } for r in results])


# Test dense search
print("DENSE SEARCH (e5-large semantic)")
print("=" * 55)

queries = [
    "dark psychological thriller about identity",
    "space exploration science fiction",
    "romantic comedy set in New York",
    "animated movie for children about friendship",
    "dream within a dream reality bending thriller",
    "wizard school magic friendship",
    "superhero billionaire flying suit",
]

for q in queries:
    print(f"\nQuery: '{q}'")
    results = dense_search(q, top_k=5)
    print(results[['title', 'score',
                   'year']].to_string(index=False))

DENSE SEARCH (e5-large semantic)

Query: 'dark psychological thriller about identity'
          title  score  year
       Identity 0.8371  2003
Double Identity 0.8294  2009
           2:13 0.8149  2009
  The Dark Past 0.8125  1948
   The Nameless 0.8111  1999

Query: 'space exploration science fiction'
            title  score  year
        Explorers 0.8234  1985
Conquest of Space 0.8208  1955
   Rocketship X-M 0.8199  1950
 Journey to Space 0.8171  2015
  A Space Program 0.8164  2015

Query: 'romantic comedy set in New York'
             title  score  year
New York, New York 0.8448  1977
       Coming Soon 0.8439  1999
          New York 0.8409  2009
         Manhattan 0.8408  1979
   Romantic Comedy 0.8400  1983

Query: 'animated movie for children about friendship'
                title  score  year
The Little Polar Bear 0.8181  2001
                Gooby 0.8163  2009
         True Friends 0.8161  1954
      Robinson Crusoe 0.8156  2016
               Friend 0.8153  2015

Query: 'dr

In [27]:
# Sparse Search (BM25)

def sparse_search(query: str,
                  top_k: int = 10) -> pd.DataFrame:
    """
    Keyword search using BM25/TF-IDF sparse vectors.
    Best for exact terms — actor names, titles.
    """
    # Transform query to sparse vector
    query_sparse = tfidf.transform([query])
    indices      = query_sparse.indices.tolist()
    values       = query_sparse.data.tolist()

    if not indices:
        return pd.DataFrame()

    results = client.search(
        collection_name = COLLECTION_NAME,
        query_vector    = NamedSparseVector(
            name   = "sparse",
            vector = SparseVector(
                indices = indices,
                values  = values,
            )
        ),
        limit        = top_k,
        with_payload = True,
    )

    return pd.DataFrame([{
        'title':    r.payload['title'],
        'score':    round(r.score, 4),
        'genres':   r.payload['genres'],
        'year':     r.payload['year'],
        'director': r.payload['director'],
    } for r in results])


# Test sparse search
print("SPARSE SEARCH (BM25 keyword)")
print("=" * 55)

for q in queries:
    print(f"\nQuery: '{q}'")
    results = sparse_search(q, top_k=5)
    if not results.empty:
        print(results[['title', 'score',
                       'year']].to_string(
                           index=False))
    else:
        print("  No results")

SPARSE SEARCH (BM25 keyword)

Query: 'dark psychological thriller about identity'
           title  score  year
  Camera Obscura 0.3459  2000
31 North 62 East 0.2754  2009
     Dark Asylum 0.2634  2001
      Mr. Brooks 0.2531  2007
      The Sublet 0.2355  2015

Query: 'space exploration science fiction'
                                    title  score  year
                                  Nuntius 0.2944  2014
                         Manhunt in Space 0.2695  1956
                           12 to the Moon 0.2653  1960
                            Magnetic Rose 0.2567  1995
Voyage to the Planet of Prehistoric Women 0.2541  1968

Query: 'romantic comedy set in New York'
                          title  score  year
               Letters to Santa 0.2902  2011
Το Ξύλο Βγήκε Από Τον Παράδεισο 0.2797  1959
                     Faintheart 0.2728  2008
                    Coming Soon 0.2493  1999
                 Happy New York 0.2460  1997

Query: 'animated movie for children about friendshi

In [28]:
# Hybrid Search (Dense + BM25)
def hybrid_search(query: str,
                  top_k: int = 10,
                  dense_weight: float = 0.7,
                  sparse_weight: float = 0.3,
                  genre_filter: str = None
                  ) -> pd.DataFrame:
    """
    Hybrid search: dense + sparse simultaneously.
    Combines semantic understanding with
    exact keyword matching.

    dense_weight  = 0.7 → favour semantic
    sparse_weight = 0.3 → some exact matching
    Sum = 1.0 (normalised)
    """
    # Dense query vector
    query_emb = embedder.encode(
        [f"query: {query}"],
        normalize_embeddings = True,
        convert_to_numpy     = True,
    )[0].tolist()

    # Sparse query vector
    query_sparse = tfidf.transform([query])
    indices = query_sparse.indices.tolist()
    values  = query_sparse.data.tolist()

    # Get more candidates from each
    # then merge + rerank
    k_candidates = top_k * 3

    # Dense results
    dense_results = client.search(
        collection_name = COLLECTION_NAME,
        query_vector    = NamedVector(
            name   = "dense",
            vector = query_emb,
        ),
        limit        = k_candidates,
        with_payload = True,
    )

    # Sparse results
    sparse_results = []
    if indices:
        sparse_results = client.search(
            collection_name = COLLECTION_NAME,
            query_vector    = NamedSparseVector(
                name   = "sparse",
                vector = SparseVector(
                    indices = indices,
                    values  = values,
                )
            ),
            limit        = k_candidates,
            with_payload = True,
        )

    # Merge scores — Reciprocal Rank Fusion (RRF)
    scores = {}
    payloads = {}

    for rank, r in enumerate(dense_results):
        pid = r.payload['movie_id']
        scores[pid]   = scores.get(pid, 0) + \
            dense_weight * r.score
        payloads[pid] = r.payload

    for rank, r in enumerate(sparse_results):
        pid = r.payload['movie_id']
        scores[pid]   = scores.get(pid, 0) + \
            sparse_weight * r.score
        payloads[pid] = r.payload

    # Sort by combined score
    sorted_results = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    return pd.DataFrame([{
        'title':    payloads[pid]['title'],
        'score':    round(score, 4),
        'genres':   payloads[pid]['genres'],
        'year':     payloads[pid]['year'],
        'director': payloads[pid]['director'],
    } for pid, score in sorted_results
      if pid in payloads])


# Test hybrid search
print("HYBRID SEARCH (dense 0.7 + sparse 0.3)")
print("=" * 55)

for q in queries:
    print(f"\nQuery: '{q}'")
    results = hybrid_search(q, top_k=5)
    if not results.empty:
        print(results[['title', 'score',
                       'year']].to_string(
                           index=False))
    else:
        print("  No results")

HYBRID SEARCH (dense 0.7 + sparse 0.3)

Query: 'dark psychological thriller about identity'
          title  score  year
    Dark Asylum 0.6462  2001
       Identity 0.5859  2003
Double Identity 0.5805  2009
           2:13 0.5704  2009
  The Dark Past 0.5688  1948

Query: 'space exploration science fiction'
            title  score  year
   Rocketship X-M 0.6398  1950
  Prince of Space 0.6350  1959
        Explorers 0.5764  1985
Conquest of Space 0.5745  1955
 Journey to Space 0.5720  2015

Query: 'romantic comedy set in New York'
             title  score  year
       Coming Soon 0.6655  1999
    Happy New York 0.6556  1997
New York, New York 0.5913  1977
          New York 0.5886  2009
         Manhattan 0.5886  1979

Query: 'animated movie for children about friendship'
                title  score  year
The Little Polar Bear 0.5727  2001
                Gooby 0.5714  2009
         True Friends 0.5713  1954
      Robinson Crusoe 0.5709  2016
               Friend 0.5707  2015

Quer

In [29]:
#Compare TF-IDF vs Dense vs Hybrid

print("RETRIEVAL METHOD COMPARISON")
print("=" * 60)
print("Query: 'movies similar to Interstellar involving space travel and time distortion")

query = "movies similar to Interstellar involving space travel and time distortion"

# TF-IDF baseline (Day 4)
tfidf_day4 = joblib.load(
    FEAT + 'tfidf_vectorizer.joblib')
tfidf_matrix_day4 = sp.load_npz(
    FEAT + 'tfidf_matrix.npz')

query_vec    = tfidf_day4.transform([query])
similarities = (tfidf_matrix_day4 @ query_vec.T)\
    .toarray().flatten()
top_idx      = np.argsort(similarities)[::-1][:5]
tfidf_results = movies.iloc[top_idx][
    ['title', 'genres_list']].copy()
tfidf_results['score'] = similarities[top_idx]\
    .round(4)

print("1. TF-IDF (Day 4 baseline):")
print(tfidf_results[['title', 'score']]\
      .to_string(index=False))

print(f"\n2. Dense e5-large:")
dense_res = dense_search(query, top_k=5)
print(dense_res[['title', 'score']]\
      .to_string(index=False))

print(f"\n3. Hybrid (dense + BM25):")
hybrid_res = hybrid_search(query, top_k=5)
print(hybrid_res[['title', 'score']]\
      .to_string(index=False))

print("""
INTERPRETATION FOR REPORT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TF-IDF: matches exact words
  → finds movies with "science" and "fiction"
  → misses semantically similar movies

e5-large dense: matches meaning
  → finds movies that FEEL like sci-fi thrillers
  → even if they use different words

Hybrid: best of both
  → semantic understanding + exact matches
  → highest quality retrieval in practice
  → 2026 benchmark winner for filtered ANN
""")

RETRIEVAL METHOD COMPARISON
Query: 'movies similar to Interstellar involving space travel and time distortion
1. TF-IDF (Day 4 baseline):
                      title  score
           Destiny in Space 0.3370
           Journey to Space 0.2662
Gayniggers from Outer Space 0.2655
    Assignment: Outer Space 0.2334
         The Dream Is Alive 0.2310

2. Dense e5-large:
                       title  score
                Interstellar 0.8144
                     Paradox 0.8059
                         Tar 0.8025
The Spaceman and King Arthur 0.8019
          The Time Travelers 0.8015

3. Hybrid (dense + BM25):
            title  score
     Interstellar 0.6215
             2010 0.6024
The Time Guardian 0.6013
          Paradox 0.5641
              Tar 0.5617

INTERPRETATION FOR REPORT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TF-IDF: matches exact words
  → finds movies with "science" and "fiction"
  → misses semantically similar movies

e5-large dense: matches meaning
  → finds mo

In [30]:
# Filtered Search Example

print("FILTERED SEARCH — Genre + Semantic")
print("=" * 55)
print("Query: 'hero saves the world'")
print("Filter: Action movies only\n")

# Filtered dense search
action_results = dense_search(
    query        = "hero saves the world",
    top_k        = 10,
    genre_filter = "Action"
)

print("Action movies matching 'hero saves world':")
print(action_results[['title', 'score', 'year']]\
      .to_string(index=False))

print("""
WHY FILTERING MATTERS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Production systems always filter candidates
before neural ranking:
  → reduce from 9,000 to 500 candidates
  → ranker (HSTU) scores only 500 not 9,000
  → 18x faster inference
  → this is the retrieve → rank pipeline
""")

FILTERED SEARCH — Genre + Semantic
Query: 'hero saves the world'
Filter: Action movies only

Action movies matching 'hero saves world':
                               title  score  year
                     Everyone's Hero 0.8019  2006
                                Hero 0.8014  2002
                       Hero at Large 0.7983  1980
                         I Am a Hero 0.7957  2015
              The Day After Tomorrow 0.7926  2004
                 Hero and the Terror 0.7901  1988
                        Man of Steel 0.7893  2013
                   Too Late the Hero 0.7890  1970
Justice League: Crisis on Two Earths 0.7889  2010
                  Last Hero in China 0.7878  1993

WHY FILTERING MATTERS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Production systems always filter candidates
before neural ranking:
  → reduce from 9,000 to 500 candidates
  → ranker (HSTU) scores only 500 not 9,000
  → 18x faster inference
  → this is the retrieve → rank pipeline



In [35]:
# Warm Cache
dense_search("warmup", top_k=10)
sparse_search("warmup", top_k=10)
hybrid_search("warmup", top_k=10)

,title,score,genres,year,director
0,The Stickup,0.5404,"[Action, Thriller]",2002,Rowdy Herrington
1,The Set-Up,0.5398,"[Crime, Drama]",1949,Robert Wise
2,Held Up,0.5395,[Comedy],1999,Steve Rash
3,The Change-Up,0.5387,[Comedy],2011,David Dobkin
4,Fired Up!,0.5348,[Comedy],2009,Will Gluck
5,Step Up Revolution,0.5335,"[Music, Drama, Romance]",2012,Scott Speer
6,Level Up,0.5333,[Thriller],2016,Adam Randall
7,Starred Up,0.5329,[Drama],2013,David Mackenzie
8,Lift me up,0.5325,[Family],2015,Mark S. Cartier
9,Fittest On Earth: A Decade Of Fitness,0.5322,[Documentary],2017,Heber Cannon


In [36]:
# Latency Benchmark
import time

print("LATENCY BENCHMARK")
print("=" * 55)

N_QUERIES = 50
test_query = "romantic drama about love and loss"

# Dense search latency
dense_times = []
for _ in range(N_QUERIES):
    start = time.time()
    dense_search(test_query, top_k=10)
    dense_times.append(
        (time.time() - start) * 1000)

# Sparse search latency
sparse_times = []
for _ in range(N_QUERIES):
    start = time.time()
    sparse_search(test_query, top_k=10)
    sparse_times.append(
        (time.time() - start) * 1000)

# Hybrid search latency
hybrid_times = []
for _ in range(N_QUERIES):
    start = time.time()
    hybrid_search(test_query, top_k=10)
    hybrid_times.append(
        (time.time() - start) * 1000)

print(f"{'Method':<15} {'p50 (ms)':<12} "
      f"{'p95 (ms)':<12} {'p99 (ms)':<12}")
print("─" * 51)

for name, times in [
    ("Dense",  dense_times),
    ("Sparse", sparse_times),
    ("Hybrid", hybrid_times),
]:
    p50 = np.percentile(times, 50)
    p95 = np.percentile(times, 95)
    p99 = np.percentile(times, 99)
    print(f"{name:<15} {p50:<12.1f} "
          f"{p95:<12.1f} {p99:<12.1f}")

print(f"""
LATENCY SLA CHECK
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Target: < 50ms p99 for retrieval

Your results show whether Qdrant meets
the production latency SLA on local hardware.
Cloud deployment (AWS) will be faster due
to colocation with the model serving layer.
""")

LATENCY BENCHMARK
Method          p50 (ms)     p95 (ms)     p99 (ms)    
───────────────────────────────────────────────────
Dense           43.3         50.2         700.0       
Sparse          2.8          5.2          8.5         
Hybrid          46.4         49.7         58.3        

LATENCY SLA CHECK
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Target: < 50ms p99 for retrieval

Your results show whether Qdrant meets
the production latency SLA on local hardware.
Cloud deployment (AWS) will be faster due
to colocation with the model serving layer.



In [37]:
#Save Results

# Save collection info
collection_info = {
    "collection":    COLLECTION_NAME,
    "n_movies":      len(movies),
    "embed_model":   "intfloat/e5-large-v2",
    "embed_dim":     EMBED_DIM,
    "sparse_vocab":  len(tfidf.vocabulary_),
    "index":         "HNSW (m=16, ef=100)",
    "search_types":  [
        "dense", "sparse", "hybrid"],
    "latency_ms": {
        "dense_p99":  round(
            np.percentile(dense_times,  99), 1),
        "sparse_p99": round(
            np.percentile(sparse_times, 99), 1),
        "hybrid_p99": round(
            np.percentile(hybrid_times, 99), 1),
    }
}

with open(PROC + 'qdrant_results.json', 'w') as f:
    json.dump(collection_info, f, indent=2)

print("✅ Qdrant collection info saved")
print(json.dumps(collection_info, indent=2))

# Log to MLflow
mlflow.set_tracking_uri(settings.MLFLOW_TRACKING_URI)
mlflow.set_experiment(settings.MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="Qdrant_hybrid"):
    mlflow.log_params({
        "embed_model":  "e5-large-v2",
        "embed_dim":    EMBED_DIM,
        "sparse_vocab": len(tfidf.vocabulary_),
        "hnsw_m":       16,
        "hnsw_ef":      100,
        "n_movies":     len(movies),
    })
    mlflow.log_metrics({
        "dense_p99_ms":  round(
            np.percentile(dense_times,  99), 1),
        "sparse_p99_ms": round(
            np.percentile(sparse_times, 99), 1),
        "hybrid_p99_ms": round(
            np.percentile(hybrid_times, 99), 1),
    })

print("\n✅ MLflow run logged")

✅ Qdrant collection info saved
{
  "collection": "movies_hybrid",
  "n_movies": 45454,
  "embed_model": "intfloat/e5-large-v2",
  "embed_dim": 1024,
  "sparse_vocab": 30000,
  "index": "HNSW (m=16, ef=100)",
  "search_types": [
    "dense",
    "sparse",
    "hybrid"
  ],
  "latency_ms": {
    "dense_p99": 700.0,
    "sparse_p99": 8.5,
    "hybrid_p99": 58.3
  }
}

✅ MLflow run logged
